## Part 5: Inference function `predict_podium(record)` (P1)

**Question the function answers:** will *this driver* finish on the podium (top 3) in *this race*? It is the same binary question as the rest of the project (`podium = 1` if `positionOrder <= 3`), asked for **one** driver in **one** race, **after qualifying and before the race starts**. It does not predict the exact finishing position and does not pick "the top three drivers of a race".

**Inputs from earlier parts (nothing is retrained here):**
* Part 4: `FINAL_MODEL` (one pipeline: preprocessing + model), `FINAL_THRESHOLD`, `FINAL_NAME`, `NUMERIC`, `CATEGORICAL`
* Part 3: `FEATURES` (the 18 pre-race features, in the order the model was trained on), `FEATURE_GROUPS`, the modeling table (`table`)
* Part 1: `POST_RACE_COLS` (columns that are only known after the race), `tables`, `outcomes`

Sections: 5.1) objects check, 5.2) the function, 5.3) Example A (complete record), 5.4) Example B (partially missing record), 5.5) checks, 5.6) limitations.

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
# In main.ipynb, Parts 1-4 run just before this part and define every object used below.
# Locally we get the same objects by running P4's notebook first (it runs P3's notebook, which runs load_clean()).
import sys
sys.path.append("../src")
from f1_clean import POST_RACE_COLS
from IPython.utils.capture import capture_output
with capture_output():          # hide Part 3 and Part 4 printouts and plots
    %run p4_models.ipynb

### 5.1 Do we have everything the function needs?
The function uses objects created in earlier parts. This cell stops with a clear message if one of them is missing (for example when this part is run before Part 4), and checks that the model really expects our 18 features in our order.

In [ ]:
import numpy as np
import pandas as pd

needed = ["FINAL_MODEL", "FINAL_THRESHOLD", "FINAL_NAME", "FEATURES", "FEATURE_GROUPS", "NUMERIC", "CATEGORICAL",
          "POST_RACE_COLS", "table", "tables", "outcomes"]
absent = [name for name in needed if name not in globals()]
assert not absent, f"run Parts 1-4 first, missing objects: {absent}"

prep = FINAL_MODEL.named_steps["prep"] if hasattr(FINAL_MODEL, "named_steps") else FINAL_MODEL.prep
assert list(prep.feature_names_in_) == list(FEATURES), "the model was trained on different columns than FEATURES"
assert not (set(FEATURES) & (set(POST_RACE_COLS) | {"podium"})), "a post-race column is in FEATURES"
print(f"final model: {FINAL_NAME}, threshold {FINAL_THRESHOLD}, {len(FEATURES)} features, categorical: {CATEGORICAL}")

### 5.2 The function

**Input (`record`):** a dictionary (or a pandas Series) with the pre-race features of one driver in one race, named exactly like the columns in `FEATURES`. A feature that is not known can be **left out** or set to `None` / `NaN`.

**What the function does, step by step:**
1. **Leakage guard:** if the record contains a post-race column (`positionOrder`, `points`, `status`, `laps`, ... or the target `podium`) it stops with an error. Those columns are never allowed in a pre-race prediction.
2. **Build one row** with exactly the `FEATURES` columns in the training order. Anything missing becomes `NaN`.
3. **Fix the types** (numbers as numbers, `era` as text) so the row looks like the training rows to the pipeline.
4. **Call `FINAL_MODEL.predict_proba`.** The pipeline already contains the preprocessing fitted on the training years (median imputation + missing-value flags, standardization, one-hot `era`), so nothing is re-implemented or re-fitted here. The model is **not** retrained.
5. **Decide:** `podium_prediction = 1` if the probability is at least `FINAL_THRESHOLD` (the threshold chosen on validation in 4.8/4.9, never on the test set).

**Output:** a dictionary with the podium probability, the 0/1 decision, the threshold, the model name, the features that were missing and any keys in the record that were ignored (this catches typos such as `"gird"`).

In [ ]:
def predict_podium(record, model=None, threshold=None):
    """Will this driver finish on the podium (top 3) in this race? Uses pre-race information only.

    record    : dict or pandas Series with the pre-race features (FEATURES) of ONE driver in ONE race.
                A feature may be left out or set to None / NaN: the pipeline then fills it with the training
                median and switches on the missing-value flag (see 4.2).
    model     : fitted pipeline with predict_proba (default: FINAL_MODEL from Part 4)
    threshold : probability from which we say "podium" (default: FINAL_THRESHOLD, chosen on validation)
    """
    model = FINAL_MODEL if model is None else model
    threshold = FINAL_THRESHOLD if threshold is None else threshold
    if isinstance(record, pd.DataFrame):                       # a one-row table is accepted too
        assert len(record) == 1, "predict_podium takes ONE driver-race record"
        record = record.iloc[0]

    leaky = set(record.keys()) & (set(POST_RACE_COLS) | {"podium"})
    if leaky:                                                  # 1) leakage guard
        raise ValueError(f"post-race information is not allowed in a pre-race prediction: {sorted(leaky)}")

    values = {}                                                # 2) one row, FEATURES order, missing -> NaN
    for f in FEATURES:
        v = record.get(f, np.nan)
        values[f] = np.nan if pd.isna(v) else v
    X = pd.DataFrame([values], columns=FEATURES)
    for f in NUMERIC:                                          # 3) types like the training rows
        X[f] = X[f].astype(float)
    for f in CATEGORICAL:
        X[f] = X[f].astype(object)

    proba = float(model.predict_proba(X)[0, 1])                # 4) preprocessing + model (already fitted)
    return {
        "podium_probability": proba,
        "podium_prediction": int(proba >= threshold),          # 5) yes / no at the validation threshold
        "threshold": threshold,
        "model": FINAL_NAME,
        "missing_features": [f for f in FEATURES if pd.isna(values[f])],
        "ignored_keys": sorted(set(record.keys()) - set(FEATURES)),
    }

### 5.3 Example A: a complete record
A **real** driver-race from the validation years (2020-2021), chosen by a fixed rule, not by hand: the first validation row in which all 18 features are known. This is also the row used in P4's hand-off check (4.13), so the numbers can be compared.

The finishing position is looked up **after** the prediction, only to tell the story. It is post-race information and never goes into the function.

In [ ]:
def describe(row):
    driver = tables["drivers"].set_index("driverId").loc[row["driverId"], "surname"]
    race = tables["races"].set_index("raceId").loc[row["raceId"], "name"]
    return f"{driver}, {race} {row['year']}"

def actual_finish(row):          # post-race, for the story only
    return int(outcomes.set_index(["raceId", "driverId"]).loc[(row["raceId"], row["driverId"]), "positionOrder"])

complete_rows = table[(table["split"] == "validation") & table[FEATURES].notna().all(axis=1)]
row_A = complete_rows.iloc[0]
record_A = row_A[FEATURES].to_dict()               # the exact input: 18 pre-race features

print("Example A:", describe(row_A))
print("input record:", record_A)
result_A = predict_podium(record_A)
print("output:", result_A)
print(f"-> {result_A['podium_probability']:.1%} podium probability; decision at threshold {result_A['threshold']}: "
      f"{'PODIUM' if result_A['podium_prediction'] else 'no podium'}")
print("(story only, known after the race) actual finish: P" + str(actual_finish(row_A)))

**How the record reaches the model:** the 18 values are put into one row in the training order; the pipeline then (1) leaves them as they are because nothing is missing (the missing-value flags are 0 before scaling), (2) standardizes the numeric values with the **training** means and standard deviations, (3) one-hot encodes `era`, and (4) the logistic regression turns the weighted sum into a probability with the sigmoid. The same steps ran on every training and validation row in Part 4.

**Reading the output:** the function ran successfully and returns a probability and a decision. That only shows the function works. Whether the model is *accurate* is not shown by one record (a single correct prediction can be luck): the evidence is the validation and test metrics of Part 4 (PR-AUC about 0.74 on validation and 0.71 on test, over hundreds of rows).

### 5.4 Example B: a partially missing record
A **real** driver-race in which information is missing for a natural reason, again chosen by a fixed rule: the first validation row of a driver's **very first Grand Prix** (`driver_form5` is empty because there is no earlier race). It is also the first round of the season, so the championship standings before the race do not exist yet. Nothing was deleted by hand.

The fields that are not known are **left out of the dictionary** completely, which is how a real incomplete record would arrive.

In [ ]:
first_race_rows = table[(table["split"] == "validation") & table["driver_form5"].isna()]
row_B = first_race_rows.iloc[0]
record_B = {k: v for k, v in row_B[FEATURES].to_dict().items() if pd.notna(v)}      # unknown fields are left out

print("Example B:", describe(row_B))
print("input record (", len(record_B), "of", len(FEATURES), "features ):", record_B)
result_B = predict_podium(record_B)
print("output:", result_B)
print(f"-> {result_B['podium_probability']:.1%} podium probability; decision at threshold {result_B['threshold']}: "
      f"{'PODIUM' if result_B['podium_prediction'] else 'no podium'}")
print("(story only, known after the race) actual finish: P" + str(actual_finish(row_B)))

# what the pipeline's imputer does with the missing fields: fill value (training median) + missing-value flag
imputer = prep.named_transformers_["num"].named_steps["simpleimputer"]
numeric_in = pd.DataFrame([{f: record_B.get(f, np.nan) for f in NUMERIC}])
filled = imputer.transform(numeric_in)[0]                       # 17 numeric values, then one 0/1 flag per feature that had gaps
what_happened = pd.DataFrame({"input": numeric_in.iloc[0], "after imputing": filled[:len(NUMERIC)], "missing flag": 0})
what_happened.iloc[imputer.indicator_.features_, 2] = filled[len(NUMERIC):].astype(int)
display(what_happened[numeric_in.iloc[0].isna()].round(2))

**How the missing values are handled:** nothing new was written for this. The pipeline from 4.2 fills each empty numeric value with the **median of the training years** and adds a **missing-value flag** (1 = "this value was missing") for every feature that had gaps in training (`qual_position`, `best_qual_s`, `gap_to_pole_pct`, the four standings columns and the two form columns). The model therefore sees both a neutral filler value and the fact that it was a filler. Standings were empty in round 1 of every season (483 of the 6,781 training rows) and driver form for a driver's first race (75 rows), so the model has seen these gaps many times.

**Assumptions and limitations:**
* The filler is a *typical* value, not the true one. The result is an estimate based on the information that exists, and it is less reliable than for a complete record.
* A feature that **never** had gaps in training (`grid`, `pit_lane_flag`, `reached_q2`, `reached_q3`, `qual_format`, `circuit_starts_before`, `circuit_podiums_before`, `home_flag`, `era`) has no flag column. If one of those is missing, it is silently replaced by the median (for example `grid` becomes 11). 5.5 shows what that does.
* Missing **qualifying** values are the dangerous case (P4's limitation in 4.12): in the training years a missing qualifying time almost always meant a driver who set no time, so the model reads "missing" as a bad sign.

### 5.5 Checks
Four checks that the function is consistent with the model and the leakage rules. They are not accuracy checks.

In [ ]:
# Check 1: record by record through predict_podium == the whole validation table through FINAL_MODEL at once
val_rows = table[table["split"] == "validation"]
direct = FINAL_MODEL.predict_proba(val_rows[FEATURES])[:, 1]
via_function = np.array([predict_podium(r[FEATURES])["podium_probability"] for _, r in val_rows.iterrows()])
print(f"Check 1: {len(val_rows)} validation rows, largest difference between the two routes: {np.abs(direct - via_function).max():.2e}")
assert np.allclose(direct, via_function, atol=1e-9)

# Check 2: post-race columns are refused
for bad in ["positionOrder", "points", "status", "laps", "podium"]:
    try:
        predict_podium({**record_A, bad: 1})
        raise AssertionError(f"{bad} was NOT blocked")
    except ValueError:
        pass
print("Check 2: post-race columns are refused:", ["positionOrder", "points", "status", "laps", "podium"])

# Check 3: 'left out', None, NaN and pd.NA mean the same thing
variants = {"left out": {k: v for k, v in record_A.items() if k != "qual_position"},
            "None": {**record_A, "qual_position": None}, "NaN": {**record_A, "qual_position": np.nan},
            "pd.NA": {**record_A, "qual_position": pd.NA}}
probs = {name: predict_podium(rec)["podium_probability"] for name, rec in variants.items()}
print("Check 3:", {k: round(v, 6) for k, v in probs.items()})
assert len(set(probs.values())) == 1

# Check 4: what happens to Example A when one whole feature group is unknown (a limitation, not a bug)
rows = [{"unknown group": "none (complete record)", "podium probability": result_A["podium_probability"]}]
for group, cols in FEATURE_GROUPS.items():
    res = predict_podium({k: v for k, v in record_A.items() if k not in cols})
    rows.append({"unknown group": group, "podium probability": res["podium_probability"]})
display(pd.DataFrame(rows).round(3))

**What we see.** Checks 1-3 pass: the function gives the same probability as the model called directly on every validation row (differences of about 1e-16), it refuses post-race columns, and every way of saying "unknown" gives the same answer.

Check 4 shows how differently the groups behave for this pole-sitter (73.7% when complete):
* **Standings, form, circuit:** small change (0.64-0.71). These gaps are normal in the training years (round 1, new drivers), so the model knows how to read them.
* **Qualifying:** collapses to about 2%. Qualifying information was missing in only 20 of the 6,781 training rows, almost always for a driver who set no time, so the model reads "missing" as a very bad sign. Even `qual_position` alone missing gives about 0.05% (Check 3).
* **Grid:** drops to 0.39. `grid` never had gaps in training, so there is no flag; it is silently replaced by the median (11) while the other qualifying columns still say P1.
* **Context (`home_flag`, `era`):** *rises* to 0.91. A missing `era` gives zeros in the era columns, exactly like the unseen `ground_effect` era in 4.12, which pushes every probability up.

So treat predictions for records with missing qualifying, grid or era information as unreliable.

### 5.6 Limitations of the function
* **Input is the 18 engineered features, not raw race data.** The features that need history (form, standings, circuit history, `home_flag`, `gap_to_pole_pct`) must be computed from earlier races, as P3 did, before the function is called. The function does not build them for a brand-new race.
* **It cannot be better than the model** in Part 4: logistic regression, PR-AUC 0.71 on test, F1 0.58 on test (too many false alarms in the unseen `ground_effect` era, see 4.10 and 4.12).
* **`era`:** the model only knows `v8` and `hybrid`; a record from 2022 onwards (`ground_effect`) gets zeros in the era columns, which pushes its probability up (4.12). An `era` value the model does not know (`ground_effect`, a typo, `None`) is silently treated the same way. P4 recommends removing `era`; `predict_podium` reads `FEATURES`, so it follows automatically if the feature list changes.
* **One record, one probability:** the function never compares drivers in the same race, so it can say "podium" for more or fewer than three drivers.
* **The decision uses the F1-optimal threshold** from validation; another use of the prediction may need a different threshold.